In [1]:
from collections import Counter

# 你给的表里明确列出的字符集合（不含“サロゲートペア文字一覧”）
CIRCLED_NUMBERS = set("①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮⑯⑰⑱⑲⑳")
ROMAN_NUMERALS = set("ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩⅪⅫ")
UNIT_SIGNS = set("㍉㌔㌢㍍㌘㌧㌃㌶㍑㍗㌍㌦㌣㌫㍊㌻㎜㎝㎞㎎㎏㏄㎡")
ERA_NAMES = set("㍾㍽㍼㍻㋿")
ENCLOSED_TEXT = set("㊤㊥㊦㊧㊨㈱㈲㈹")
ABBREVIATIONS = set("№℡㏍")
MATH_SIGNS = set("≒≡∫§Σ√⊥∠∟⊿∵∩∪")
IBM_UNICODE = set("－―∥〜¢£¬")
OTHERS = set("〝〟")

# 常见“简体专用字”（保守集合，可按业务继续扩充）
SIMPLIFIED_ONLY_CHARS = set("长纳")

# 半角カタカナ（表中の対象を Unicode ブロックでカバー）
HALFWIDTH_KATAKANA_START = 0xFF61
HALFWIDTH_KATAKANA_END = 0xFF9F


def _category_of_char(ch: str) -> str | None:
    cp = ord(ch)

    if ch in CIRCLED_NUMBERS:
        return "囲み英数字（丸文字）"
    if ch in ROMAN_NUMERALS:
        return "ローマ数字"
    if ch in UNIT_SIGNS:
        return "単位記号"
    if ch in ERA_NAMES:
        return "年号"
    if ch in ENCLOSED_TEXT:
        return "囲み文字"
    if ch in ABBREVIATIONS:
        return "省略文字"
    if ch in MATH_SIGNS:
        return "数字記号"
    if HALFWIDTH_KATAKANA_START <= cp <= HALFWIDTH_KATAKANA_END:
        return "半角カタカナ"
    if ch in IBM_UNICODE:
        return "IBM-Unicode"
    if ch in OTHERS:
        return "その他"
    if ch in SIMPLIFIED_ONLY_CHARS:
        return "简体字（可能非日语汉字）"

    # サロゲートペア文字一覧は「一例」なので、非BMP文字（U+10000以上）を対象とする
    if cp > 0xFFFF:
        return "サロゲートペア文字"

    return None


def check_machine_dependent_text(
    text: str, is_replace: bool, replace_text: str = ""
) -> dict:
    """
    输入:
      text: 判定対象文字列
      is_replace: True なら対象文字を replace_text で置換
      replace_text: 置換文字列

    输出:
      contains: 是否包含目标文字
      total_count: 目标文字总数
      counts_by_category: 分类统计
      matches: 命中详情
      marked_text: 将命中字符标记为 [[字]] 的字符串
      output_text: 替换后字符串（is_replace=True）或原文
    """
    matches = []
    marked_parts = []
    output_parts = []

    for idx, ch in enumerate(text):
        category = _category_of_char(ch)
        if category is not None:
            matches.append(
                {
                    "index": idx,
                    "char": ch,
                    "codepoint": f"U+{ord(ch):04X}",
                    "category": category,
                }
            )
            marked_parts.append(f"[[{ch}]]")
            output_parts.append(replace_text if is_replace else ch)
        else:
            marked_parts.append(ch)
            output_parts.append(ch)

    counts = Counter(m["category"] for m in matches)

    return {
        "contains": len(matches) > 0,
        "total_count": len(matches),
        "counts_by_category": dict(counts),
        "matches": matches,
        "marked_text": "".join(marked_parts),
        "output_text": "".join(output_parts),
    }


# ===== 你的输入参数（按需修改）=====
text = """

"""
is_replace = True
replace_text = "*"

result = check_machine_dependent_text(text, is_replace, replace_text)
result

{'contains': False,
 'total_count': 0,
 'counts_by_category': {},
 'matches': [],
 'marked_text': '\n\n',
 'output_text': '\n\n'}